# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Load Dataset

This section loads the provided dataset files from the `data` folder, including the evidence corpus, training claims, and development claims. We also print the number of evidence passages and claims to verify that the files are loaded correctly.

In [2]:
import json
from pathlib import Path

DATA_DIR = Path("data")

with open(DATA_DIR / "evidence.json", "r") as f:
    evidence = json.load(f)

with open(DATA_DIR / "train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(DATA_DIR / "dev-claims.json", "r") as f:
    dev_claims = json.load(f)

print("Evidence:", len(evidence))
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))

first_eid = list(evidence.keys())[0]
print(first_eid, evidence[first_eid])

first_cid = list(dev_claims.keys())[0]
print(first_cid, dev_claims[first_cid])

Evidence: 1208827
Train claims: 1228
Dev claims: 154
evidence-0 John Bennet Lawes, English entrepreneur and agricultural scientist
claim-752 {'claim_text': '[South Australia] has the most expensive electricity in the world.', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-67732', 'evidence-572512']}


## 1.2 Build TF-IDF Evidence Index

This section builds a TF-IDF index over all evidence passages. Each evidence text is converted into a sparse vector representation. The retriever later compares each claim vector against this evidence matrix to find the most relevant evidence passages.

We use unigram and bigram features, keep stop words, apply sublinear term frequency scaling, and remove extremely rare terms with `min_df=2`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#把evidence转成list
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

print("Total evidence:", len(evidence_texts))
print("Example:", evidence_texts[0])

# TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    ngram_range=(1, 2),
    max_features=500000,
    sublinear_tf=True,
    min_df=2
)

evidence_matrix = vectorizer.fit_transform(evidence_texts)

print("TF-IDF matrix shape:", evidence_matrix.shape)

Total evidence: 1208827
Example: John Bennet Lawes, English entrepreneur and agricultural scientist
TF-IDF matrix shape: (1208827, 500000)


## 1.3 Retrieve Top-k Evidence

This section defines a retrieval function for a single claim. The claim is transformed into the same TF-IDF vector space as the evidence corpus. Cosine similarity is then used to rank all evidence passages, and the top-k evidence IDs are returned.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_top_k(claim_text, k=5):
    claim_vec = vectorizer.transform([claim_text])
    scores = cosine_similarity(claim_vec, evidence_matrix).flatten()

    top_indices = np.argsort(scores)[::-1][:k]

    return [evidence_ids[i] for i in top_indices]

In [ ]:
# test一个claim
claim_id = list(dev_claims.keys())[0]
claim_text = dev_claims[claim_id]["claim_text"]

print("Claim:", claim_text)

retrieved = retrieve_top_k(claim_text, k=5)

print("Retrieved evidence IDs:", retrieved)
print("Gold evidence:", dev_claims[claim_id]["evidences"])

Claim: [South Australia] has the most expensive electricity in the world.
Retrieved evidence IDs: ['evidence-67732', 'evidence-572512', 'evidence-509525', 'evidence-32494', 'evidence-147175']
Gold evidence: ['evidence-67732', 'evidence-572512']


In [10]:
def show_retrieval_case(claim_id, retrieved_ids):
    claim = dev_claims[claim_id]["claim_text"]
    gold_ids = dev_claims[claim_id]["evidences"]

    print("=" * 80)
    print("Claim:")
    print(claim)

    print("\nGold evidence:")
    for eid in gold_ids:
        print(f"\n{eid}:")
        print(evidence[eid])

    print("\nRetrieved evidence:")
    for eid in retrieved_ids:
        print(f"\n{eid}:")
        print(evidence[eid])

show_retrieval_case(claim_id, retrieved)

Claim:
[South Australia] has the most expensive electricity in the world.

Gold evidence:

evidence-67732:
[citation needed] South Australia has the highest retail price for electricity in the country.

evidence-572512:
"South Australia has the highest power prices in the world".

Retrieved evidence:

evidence-67732:
[citation needed] South Australia has the highest retail price for electricity in the country.

evidence-572512:
"South Australia has the highest power prices in the world".

evidence-509525:
It was the most expensive record I ever made.

evidence-32494:
This is the most expensive way to call internationally.

evidence-147175:
Manhattan 's real estate market is among the most expensive in the world.


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*